# 01 - Data Cleaning: MRL Eye Dataset (Driver Drowsiness Detection)

**Phase 1, step 1.** Goal: turn the raw MRL Eye image folder into a clean, trustworthy table (`metadata_clean.csv`) that the EDA and preprocessing notebooks can use.

**What we check, and why**

| Step | Why it matters |
|---|---|
| Filename format / missing values | The labels (eye state) and subject ID are stored *in the filename*. A malformed name means a missing or wrong label. |
| Invalid values | Each field has a fixed set of legal values; anything else is a data-entry error. |
| Unreadable / corrupt images | They would crash training later. |
| Duplicates | Identical images can leak between train and test and inflate accuracy. |
| Outliers | Blank or near-constant images carry no eye information. |
| Inconsistent formats | Different sizes, colour modes and file extensions must be standardised before modeling. |

**MRL filename format:** `s0001_00001_0_0_0_0_0_01.png` = `subject_imageID_gender_glasses_eyeState_reflections_lighting_sensor`

| Field | Values |
|---|---|
| gender | 0 = male, 1 = female |
| glasses | 0 = no, 1 = yes |
| eye state | **0 = closed, 1 = open** (original MRL coding) |
| reflections | 0 = none, 1 = small, 2 = big |
| lighting | 0 = bad, 1 = good |
| sensor | 1, 2, 3 (three different cameras) |

**Target variable:** `closed` (we flip the MRL coding so that **1 = closed, 0 = open**; "closed" is the class we want to detect for drowsiness).

In [ ]:
import os, hashlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

sns.set_theme(style='whitegrid')
pd.set_option('display.width', 160)

# ---- paths (relative to the notebooks/ folder) ----
DATA_DIR = os.environ.get('MRL_DIR', '../data/raw/mrl')   # folder with the MRL images (subfolders allowed)
PROC_DIR = '../data/processed'
IMG_DIR  = '../images'
os.makedirs(PROC_DIR, exist_ok=True); os.makedirs(IMG_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(IMG_DIR, name), dpi=150, bbox_inches='tight')

cleaning_log = []   # (step, rows_removed, rows_remaining)
def log(step, removed, remaining):
    cleaning_log.append(dict(step=step, rows_removed=int(removed), rows_remaining=int(remaining)))
    print(f'{step}: removed {removed}, remaining {remaining}')

## 1. Find all files
We walk the folder recursively and keep track of file extensions (case-insensitive). Non-image files are reported but ignored.

In [ ]:
IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp'}
all_files = [os.path.join(r, f) for r, _, fs in os.walk(DATA_DIR) for f in fs]
assert all_files, f'No files found in {DATA_DIR}. Download the dataset first (see README).'

ext = pd.Series([os.path.splitext(f)[1] for f in all_files])
print('Extensions found (raw):'); print(ext.value_counts().to_string())

paths = [f for f in all_files if os.path.splitext(f)[1].lower() in IMG_EXT]
print(f'\n{len(all_files)} files, {len(paths)} are images, {len(all_files)-len(paths)} ignored (non-image).')
log('Raw image files found', 0, len(paths))

## 2. Parse filenames into a metadata table (format + missing-value check)
We split every filename on `_`. A valid name has **exactly 8 fields**. Numeric fields that cannot be converted become `NaN` (= missing/invalid).

In [ ]:
COLS = ['subject', 'image_id', 'gender', 'glasses', 'eye_state', 'reflections', 'lighting', 'sensor']
rows = []
for p in paths:
    stem = os.path.splitext(os.path.basename(p))[0].strip()
    parts = stem.split('_')
    rec = {'path': p, 'n_fields': len(parts)}
    if len(parts) == 8:
        rec.update(dict(zip(COLS, parts)))
    rows.append(rec)
meta = pd.DataFrame(rows)

# consistent formats: lower-case subject IDs, numeric types for the coded fields
meta['subject'] = meta['subject'].str.lower().str.strip()
NUM = COLS[1:]
for c in NUM:
    meta[c] = pd.to_numeric(meta[c], errors='coerce')

print('Filenames with wrong number of fields:', (meta.n_fields != 8).sum())
print('\nMissing values per column:'); print(meta[COLS].isna().sum().to_string())
meta[meta[COLS].isna().any(axis=1)].head()

In [ ]:
# validity of every coded field
ALLOWED = {'gender': {0, 1}, 'glasses': {0, 1}, 'eye_state': {0, 1},
           'reflections': {0, 1, 2}, 'lighting': {0, 1}, 'sensor': {1, 2, 3}}
valid = (meta.n_fields == 8) & meta.subject.str.match(r'^s\d+$', na=False) & meta[NUM].notna().all(axis=1)
for c, ok in ALLOWED.items():
    valid &= meta[c].isin(ok)

print('Invalid / malformed rows:', (~valid).sum())
print(meta.loc[~valid, ['path']].assign(file=lambda d: d.path.map(os.path.basename)).file.head(10).to_string())

before = len(meta)
meta = meta[valid].copy().reset_index(drop=True)
for c in NUM: meta[c] = meta[c].astype(int)
log('Dropped malformed / missing / invalid-value filenames', before - len(meta), len(meta))

## 3. Create the target label and inspect every image
`closed = 1 - eye_state` (so 1 = closed, 0 = open). For every image we also record width, height, colour mode, mean brightness, contrast (std of pixels) and two hashes (file bytes and pixel content) used to detect duplicates. Unreadable files are flagged.

In [ ]:
meta['closed'] = 1 - meta['eye_state']

def inspect(p):
    rec = {'readable': True}
    try:
        with open(p, 'rb') as f: raw = f.read()
        rec['file_md5'] = hashlib.md5(raw).hexdigest()
        rec['file_kb'] = len(raw) / 1024
        with Image.open(p) as im:
            rec['mode'], rec['format'] = im.mode, im.format
            rec['width'], rec['height'] = im.size
            g = np.asarray(im.convert('L'))
        rec['mean_intensity'] = float(g.mean())
        rec['std_intensity'] = float(g.std())
        rec['pixel_md5'] = hashlib.md5(g.tobytes() + str(g.shape).encode()).hexdigest()
    except Exception as e:
        rec['readable'] = False
        rec['error'] = str(e)[:80]
    return rec

info = pd.DataFrame([inspect(p) for p in tqdm(meta.path)])
meta = pd.concat([meta, info], axis=1)
print('Unreadable / corrupt images:', (~meta.readable).sum())
meta.loc[~meta.readable, ['path', 'error']].head()

In [ ]:
before = len(meta)
meta = meta[meta.readable].drop(columns=['readable', 'error'], errors='ignore').reset_index(drop=True)
log('Dropped unreadable / corrupt images', before - len(meta), len(meta))

## 4. Duplicates
Two checks:
1. **Exact file duplicates** (same bytes).
2. **Same pixels** after grayscale conversion. If a duplicate group contains *different labels* the label is untrustworthy, so the whole group is removed. Otherwise we keep the first copy.

In [ ]:
print('Exact file-byte duplicates   :', meta.duplicated('file_md5').sum())
print('Same-pixel duplicates        :', meta.duplicated('pixel_md5').sum())

grp = meta.groupby('pixel_md5')['closed'].nunique()
conflict_hashes = grp[grp > 1].index
print('Duplicate groups with conflicting labels:', len(conflict_hashes))

before = len(meta)
meta = meta[~meta.pixel_md5.isin(conflict_hashes)]
log('Dropped duplicate groups with conflicting labels', before - len(meta), len(meta))

before = len(meta)
meta = meta.drop_duplicates('pixel_md5', keep='first').reset_index(drop=True)
log('Dropped duplicate images (kept first copy)', before - len(meta), len(meta))

## 5. Inconsistent formats
We look at image size, colour mode and file format. MRL images come in slightly different sizes, so we **do not delete** them. They are standardised later (grayscale, 64x64) in notebook 03. Only absurdly tiny images are removed in the outlier step.

In [ ]:
print('Colour modes:'); print(meta['mode'].value_counts().to_string())
print('\nFile formats:'); print(meta['format'].value_counts().to_string())
print('\nImage size summary:'); print(meta[['width', 'height']].describe().round(1).to_string())
print('\nDistinct sizes:', meta.groupby(['width', 'height']).ngroups)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(meta.width, bins=30, ax=ax[0], color='steelblue'); ax[0].set_title('Image width (px)')
sns.histplot(meta.height, bins=30, ax=ax[1], color='darkorange'); ax[1].set_title('Image height (px)')
plt.tight_layout(); savefig('clean_01_image_sizes.png'); plt.show()

**What this tells us:** the histograms show how much the image sizes vary. Because a CNN needs a fixed input size, every image will be resized to one common size in notebook 03.

## 6. Outliers
* **Blank / near-constant images** (pixel std below `MIN_STD`): no visible eye, so they are **removed**.
* **Tiny images** (shorter side below `MIN_SIDE`): too little detail, **removed**.
* **Extreme brightness** (outside 1.5 x IQR of mean intensity): **flagged but kept**. Very dark or very bright images are real lighting conditions (MRL has a "bad lighting" group) and the model must learn to handle them.

In [ ]:
MIN_STD, MIN_SIDE = 3.0, 20

q1, q3 = meta.mean_intensity.quantile([.25, .75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
meta['brightness_outlier'] = (meta.mean_intensity < lo) | (meta.mean_intensity > hi)
print(f'Brightness fence: [{lo:.1f}, {hi:.1f}] -> {meta.brightness_outlier.sum()} flagged (kept)')
print('Blank / near-constant images (std <', MIN_STD, '):', (meta.std_intensity < MIN_STD).sum())
print('Tiny images (min side <', MIN_SIDE, '):', (meta[['width', 'height']].min(axis=1) < MIN_SIDE).sum())

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(x=meta.mean_intensity, ax=ax[0], color='lightsteelblue'); ax[0].set_title('Mean brightness per image')
sns.boxplot(x=meta.std_intensity, ax=ax[1], color='lightsalmon'); ax[1].set_title('Contrast (pixel std) per image')
sns.scatterplot(data=meta.sample(min(5000, len(meta)), random_state=0), x='mean_intensity', y='std_intensity',
                hue='brightness_outlier', s=8, ax=ax[2]); ax[2].set_title('Brightness vs contrast')
plt.tight_layout(); savefig('clean_02_outlier_boxplots.png'); plt.show()

In [ ]:
bad = (meta.std_intensity < MIN_STD) | (meta[['width', 'height']].min(axis=1) < MIN_SIDE)
before = len(meta)
meta = meta[~bad].reset_index(drop=True)
log('Dropped blank / tiny images (outliers)', before - len(meta), len(meta))

**What this tells us:** the boxplots show the spread of brightness and contrast. Points beyond the whiskers are unusually dark/bright images. We keep them because they represent genuine lighting variation.

## 7. Save the cleaned dataset and the cleaning log
We store the path **relative to `DATA_DIR`** so the table works on any machine.

In [ ]:
meta['rel_path'] = meta.path.map(lambda p: os.path.relpath(p, DATA_DIR))
keep = ['rel_path', 'subject', 'image_id', 'gender', 'glasses', 'eye_state', 'reflections', 'lighting', 'sensor',
        'closed', 'width', 'height', 'mode', 'format', 'file_kb', 'mean_intensity', 'std_intensity', 'brightness_outlier']
clean = meta[keep].copy()
clean.to_csv(f'{PROC_DIR}/metadata_clean.csv', index=False)

log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(f'{PROC_DIR}/cleaning_log.csv', index=False)
print('Final checks -> missing values:', int(clean.isna().sum().sum()),
      '| duplicate paths:', int(clean.rel_path.duplicated().sum()),
      '| labels:', sorted(clean.closed.unique()))
log_df

In [ ]:
# Visual summary of the cleaning pipeline
steps = log_df[log_df.rows_remaining > 0]
plt.figure(figsize=(10, 4))
sns.barplot(data=steps, y='step', x='rows_remaining', color='seagreen')
plt.title('Rows remaining after each cleaning step'); plt.xlabel('rows'); plt.ylabel('')
plt.tight_layout(); savefig('clean_03_cleaning_pipeline.png'); plt.show()

## Summary of cleaning
* Parsed 8 metadata fields from every filename and checked format, missing values and legal ranges.
* Removed unreadable files, duplicate images (and any duplicate group with contradicting labels), blank/tiny images.
* Flagged (but kept) brightness outliers because they are real lighting conditions.
* Image size / colour mode inconsistencies are documented here and standardised in notebook 03.
* Output: `data/processed/metadata_clean.csv` and `data/processed/cleaning_log.csv` (see the table above for the exact counts).

**Next:** `02_eda.ipynb`